# C0 · 真实资源效率测试

这是 **Owner 手工启动** 的受限资源实验，不是 HPO 或前瞻验证。6 个逻辑任务在缓存关闭、冷/热缓存、CPU 并行及可用 CUDA 下重复，仅测量系统效果；最多 36 次 fit，不增加独立金融样本。2023–26 已消费窗口始终标为 RETROSPECTIVE。

本地先导已证明缓存数值一致；**真实 Colab、GPU 和 Compute Units 尚未验收**。内部自用按 Owner 2026-11-08 明确授权执行，不要求供应商证明文件。原 E4–E7 结果不改写。

操作：选择运行时 → 手工运行各单元格 → R1/R2 成功才测并行 → 私人 Drive 返回包交本地复核。没有远程控制、持久调度或自动晋级。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, os, json, hashlib, shutil, zipfile, io, uuid, time
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUNNER_SHA = 'f7221d5c380f9dd9507540b65bdf4e4133392bcd'
JOB_NAME = 'c0-us73-resources-20261108-v3-r3'
RESUME_ARCHIVE = None  # 可填上次 checkpoint.zip 的完整 Drive 路径
RESUME_SHA256 = None   # 填上次输出的 ZIP SHA256，不能猜测或自动选择 latest
TRY_CUDA = True        # 仅真实 CUDA 硬件且训练器实际采用 CUDA 才计为 GPU 测量
LOCAL_BASE = Path('D:/Documents/AlphaEnginePrivate/Launch-Research/2026-11-08/research')
LOCAL_CODE = Path('D:/Documents/GitHub/alphaengine/.worktrees/research-c0-20261108')
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
CU_BEFORE = None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AlphaEngine')
    JOB = ROOT/'03_Data/Research_Exchange_P2/colab_jobs'/JOB_NAME
    answer = input('记录当前 Colab UI 剩余 Compute Units（无数据留空，不能估算）：').strip()
    CU_BEFORE = float(answer) if answer else None
    SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
    SCRATCH = Path('/content')/('alpha-c0-'+SESSION)
    SCRATCH.mkdir(exist_ok=False)
    CODE = SCRATCH/'code'
    DATA = SCRATCH/'data'
    OUTPUT = SCRATCH/'results'
    OUTPUT.mkdir()
else:
    JOB = LOCAL_BASE
    CODE = LOCAL_CODE
    DATA = LOCAL_BASE/'source-stage'
    OUTPUT = LOCAL_BASE/'local-benchmark-v3'
    SESSION = 'local-readback-no-fits'
SETUP_STARTED = time.perf_counter()
print('模式：', 'Owner Colab 实测' if IN_COLAB else '本地既有结果只读复核；不训练')

## 冻结输入与环境

源码固定到上方 40 位 SHA，数据只复用经哈希验证的 US 行情组件，候选为明确的 US73，QQQ 单列。旧 v6 暂存回执保持原字节；本轮 Owner 授权由新的 C0 spec 记录。
部署 r3（包含 r2 锁文件修复） 在安装前恢复已封存的锁文件原字节，并核对 Git 内容仅有 CRLF/LF 差异；8 个源码哈希仍严格一致，不放宽检查。计算版本与 v3 结果均不变。
缓存、线程和设备均进入记录；R1→R2→并行必须在同一实测环境。硬件改变后须新建测量 run，不能把不同机器的阶段拼成吞吐优势。

In [ ]:
def command(args, log=None, check=True):
    result = subprocess.run(list(map(str,args)), cwd=CODE if CODE.exists() else None,
                            env={**os.environ,'OMP_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1'},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if log is not None:
        log = Path(log)
        log.parent.mkdir(parents=True,exist_ok=True)
        if log.exists(): log = log.with_name(log.stem+'-attempt-'+uuid.uuid4().hex[:8]+log.suffix)
        with log.open('x',encoding='utf-8') as f: f.write(result.stdout)
    if check and result.returncode:
        raise RuntimeError(f'命令失败（{result.returncode}），查看 {log}；原日志保留')
    return result
if IN_COLAB:
    manifest = json.loads((JOB/'delivery-manifest.json').read_bytes())
    for name, entry in manifest['job_files'].items():
        assert (JOB/name).stat().st_size == entry['bytes'] and sha(JOB/name) == entry['sha256'], name
    subprocess.run([sys.executable,'-m','pip','install','uv==0.9.10'],check=True)
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/liuh886/alpha_engine.git',str(CODE)],check=True)
    command(['git','sparse-checkout','init','--cone'])
    command(['git','sparse-checkout','set','src','configs'])
    command(['git','fetch','origin',RUNNER_SHA])
    command(['git','checkout','--detach',RUNNER_SHA])
    # Preserve the frozen Windows lock bytes: Linux Git checkout normalizes CRLF.
    frozen_spec = json.loads((JOB/'c0-spec-v3.json').read_bytes())
    lock_bytes = (JOB/'uv.lock.frozen').read_bytes()
    assert hashlib.sha256(lock_bytes).hexdigest() == frozen_spec['code_sha256']['uv.lock']
    git_lock = subprocess.check_output(['git','show',RUNNER_SHA+':uv.lock'],cwd=CODE)
    assert lock_bytes.replace(b'\r\n',b'\n') == git_lock, 'Only newline representation may differ'
    (CODE/'uv.lock').write_bytes(lock_bytes)
    for relative, expected in frozen_spec['code_sha256'].items():
        assert sha(CODE/relative) == expected, 'Frozen source mismatch: '+relative
    print('Frozen source: all 8 raw SHA256 hashes PASS; lock Git content identical after LF normalization')
    command(['uv','python','install','3.12.12'],SCRATCH/'python-install.log')
    command(['uv','sync','--frozen','--no-install-project','--python','3.12.12'],SCRATCH/'uv-sync.log')
    PY = CODE/'.venv/bin/python'
    SPEC = SCRATCH/'c0-spec.json'
    SPEC.write_bytes((JOB/'c0-spec-v3.json').read_bytes())
    command([PY,'-m','src.research.v5_isolated_staging','stage','--manifest',JOB/'bundle-manifest.json',
             '--archive',JOB/'us_market_history.zip','--expected-manifest-sha256','4f8225f3c68ee0886b9b707ca140e91dcb509b739b8548fdc10f258574d3da1a',
             '--expected-archive-sha256','b4745e46fc26b0328e22de50e579ebbf1d400996258ea6ebb43c1ff739ed5a1a','--output-root',DATA],SCRATCH/'staging.log')
else:
    PY = CODE/'.venv/Scripts/python.exe'
    SPEC = JOB/'c0-spec-v3.json'
spec = json.loads(SPEC.read_bytes())
assert spec['runner_git_sha'] == RUNNER_SHA and spec['owner_internal_research_authorized'] is True
assert spec['vendor_document_review_required'] is False and len(spec['tasks']) == 6
print('源码：', RUNNER_SHA, '\nSpec SHA256：',sha(SPEC),'\n逻辑任务：',len(spec['tasks']))

## 检查点恢复

仅恢复明确选定且 SHA256 匹配的本实验结果；不读取 E4–E7，也不自动寻找旧绿包。完成的阶段按文件哈希复用，不重复 fit。
若在阶段中途被终止，不覆盖不完整阶段；保留失败包，新建 run 重测该阶段。正式中断恢复的云端验收仍需 Owner 实际演示。

In [ ]:
if IN_COLAB and RESUME_ARCHIVE:
    assert RESUME_SHA256 and sha(RESUME_ARCHIVE) == RESUME_SHA256
    with zipfile.ZipFile(RESUME_ARCHIVE) as z:
        index = json.loads(z.read('bundle-index.json'))
        assert index['spec_sha256'] == sha(SPEC) and index['runner_git_sha'] == RUNNER_SHA
        assert len(z.infolist()) <= 2000 and sum(i.file_size for i in z.infolist()) <= 256*1024**2
        for name,entry in index['files'].items():
            relative = Path(name)
            assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name
            raw = z.read(name)
            assert len(raw)==entry['bytes'] and hashlib.sha256(raw).hexdigest()==entry['sha256']
            target=OUTPUT/relative; target.parent.mkdir(parents=True,exist_ok=True)
            with target.open('xb') as f: f.write(raw)
    print('恢复完成；后续命令先校验已完成阶段与实际硬件身份。')

## R1 / R2 与 CPU、GPU 实测

关闭缓存、缓存空作用域（首个 fold 冷，其余任务可能命中）、热缓存分别记录。热缓存只指 R1 内存缓存，未清理 OS 文件缓存。
数值验收比较完整评分、逐日净/毛收益、换手、基准与实际模拟权重数组，CPU 要求逐元素一致。R2 未通过则禁止扩并行。
CPU 2/4 worker 需要实测内存预算；不足只阻断该 CPU arm。CUDA 必须由已支持的 XGBoost hist 实际采用，不能把 CPU fallback 算作 GPU。
GPU纳入可比资源收益的默认门槛在运行前冻结为：6任务完整数组全部逐元素一致；否则只保留差异与费用，不事后调宽阈值选GPU。

In [ ]:
BASE_COMMAND = [PY,'-m','src.research.c0_resource_benchmark','run','--spec',SPEC,'--data-root',DATA,'--output',OUTPUT]
def checkpoint(label):
    if not IN_COLAB: return None
    destination=ROOT/'03_Data/Research_Exchange_P2/incoming_unreviewed'/JOB_NAME/SESSION
    destination.mkdir(parents=True,exist_ok=True)
    files={p.relative_to(OUTPUT).as_posix():p.read_bytes() for p in OUTPUT.rglob('*') if p.is_file()}
    index={'schema_version':'c0_checkpoint_v1','runner_git_sha':RUNNER_SHA,'spec_sha256':sha(SPEC),
           'files':{n:{'bytes':len(b),'sha256':hashlib.sha256(b).hexdigest()} for n,b in files.items()}}
    path=destination/(label+'-'+uuid.uuid4().hex[:8]+'.zip')
    with zipfile.ZipFile(path,'x',zipfile.ZIP_DEFLATED) as z:
        for n,raw in files.items(): z.writestr(n,raw)
        z.writestr('bundle-index.json',json.dumps(index,indent=2))
    with zipfile.ZipFile(path) as z: assert z.testzip() is None
    record={'path':str(path),'sha256':sha(path),'bytes':path.stat().st_size,'source_spec_sha256':sha(SPEC),
            'review_status':'incoming_unreviewed','research_only':True,'trade_ready':False}
    with path.with_suffix('.manifest.json').open('x',encoding='utf-8') as f: json.dump(record,f,indent=2)
    print(label,record['bytes'],record['sha256'])
    return record
def measure(label,extra):
    if not IN_COLAB: return
    # A sealed completed run is read-only on resume; never overwrite its audit.
    if label=='cpu-and-gpu' and (OUTPUT/'completed-audit.json').exists():
        print('复用已完成 run；不追加 GPU arm。另测设备须新建 run。')
        args=[PY,'-m','src.research.c0_resource_benchmark','verify','--output',OUTPUT]
    else:
        args=BASE_COMMAND+extra
    result=command(args,OUTPUT/(label+'.log'),check=False)
    status='PASS' if result.returncode==0 else 'BLOCKED；查看保留日志'
    if label=='cpu-and-gpu' and (OUTPUT/'completed-audit.json').exists():
        status=json.loads((OUTPUT/'completed-audit.json').read_bytes())['status']
    record={'stage':label,'exit_code':result.returncode,'status':status,'research_only':True,'trade_ready':False}
    status_path=OUTPUT/(label+'-execution-status.json')
    if status_path.exists(): status_path=OUTPUT/(label+'-execution-status-attempt-'+uuid.uuid4().hex[:8]+'.json')
    with status_path.open('x',encoding='utf-8') as f: json.dump(record,f,indent=2)
    print(label,status,result.returncode)
    checkpoint(label)
    if result.returncode and label!='cpu-and-gpu': raise RuntimeError('保留失败，不覆盖结果；检查后用新的 run 继续')
measure('r1',['--stop-after','r1'])

In [ ]:
measure('r2',['--stop-after','r2'])

In [ ]:
if IN_COLAB:
    gpu_probe=subprocess.run(['bash','-lc','command -v nvidia-smi >/dev/null && nvidia-smi -L'],capture_output=True,text=True)
    enable_gpu=TRY_CUDA and gpu_probe.returncode==0
    print('CUDA 硬件探测：',gpu_probe.stdout.strip() if enable_gpu else 'NOT_TESTED / 未提供 CUDA 硬件')
    hardware={'probe_returncode':gpu_probe.returncode,'nvidia_smi_L':gpu_probe.stdout.strip(),
              'gpu_memory_peak_status':'NOT_MEASURED_RSS_IS_HOST_MEMORY_ONLY'}
    hardware_path=OUTPUT/'gpu-hardware.json'
    if hardware_path.exists(): assert json.loads(hardware_path.read_bytes())==hardware, 'GPU硬件身份改变，须新建测量run'
    else: hardware_path.write_text(json.dumps(hardware,indent=2),encoding='utf-8')
    measure('cpu-and-gpu',['--gpu'] if enable_gpu else [])

## 只读数值复核与资源表

核验命令不训练模型。表中 wall time 是阶段端到端时长；RSS 为 20ms 采样的进程树峰值，非严格 OS 高水位。热/冷的每任务合计应另看测量回执，不混为训练核心提速。

In [ ]:
audit_run=command([PY,'-m','src.research.c0_resource_benchmark','verify','--output',OUTPUT],check=False)
if audit_run.returncode:
    raise RuntimeError('独立数值核验失败，不能提交为通过')
audit=json.loads(audit_run.stdout)
assert audit['cpu_exact'] and audit['same_environment'] and audit['r2_cache_reproducible'], 'CPU数值/缓存/环境未通过；保留失败，不提交通过结果'
print('完整 CPU 数组一致：',audit['cpu_exact'],'；R2：',audit['r2_cache_reproducible'])
print('逻辑任务：',audit['logical_tasks'],'；已保存 fit：',audit['observed_completed_fits'],
      '；新增独立金融样本：',audit['financial_independent_samples_added'])
for name,row in audit['stages'].items():
    print(f"{name:24} {row['wall_seconds']:9.3f} 秒 / {row['sampled_tree_peak_rss_bytes']/1024**2:9.2f} MiB")
if (OUTPUT/'completed-audit.json').exists():
    completed=json.loads((OUTPUT/'completed-audit.json').read_bytes())
    print('容量阻断：',len(completed.get('capacity_blockers',[])),'；GPU：',completed.get('gpu_status'))
gpu_rows=[r for r in audit['comparisons'] if r['stage']=='gpu-cuda']
print('GPU数值准入：',all(r['exact'] for r in gpu_rows) if gpu_rows else 'NOT_TESTED')
print('Compute Units 尚需实际 Colab UI 记录，不从时长估算。')

## Compute Units 与结果返回

CU 仅记录 Owner 实际 UI 读数，保留测量方法；没有读数就是 NOT_MEASURED，不能宣称云端成本优势。
输出为私人 `incoming_unreviewed`，本地需独立复核全部文件 SHA、预测/净值/换手数组、环境与 GPU 实际设备后才能采信。不能将已消费历史称为独立 holdout。

官方说明：[XGBoost GPU](https://xgboost.readthedocs.io/en/stable/gpu/index.html)；[Colab 资源与交互限制](https://research.google.com/colaboratory/faq.html)。本 Notebook 不承诺特定 GPU 或不中断运行。

In [ ]:
if IN_COLAB:
    answer=input('记录结束时 Colab UI 剩余 Compute Units（无数据留空）：').strip()
    after=float(answer) if answer else None
    used=CU_BEFORE-after if CU_BEFORE is not None and after is not None else None
    assert used is None or used>=0, 'CU读数不一致，不能猜测修正'
    record={'before':CU_BEFORE,'after':after,'used':used,'method':'Owner reported actual Colab UI balance',
            'status':'OWNER_OBSERVED_NOT_INDEPENDENTLY_ATTESTED' if used is not None else 'NOT_MEASURED',
            'includes_setup':True,'excluded_initial_drive_mount':True,'setup_and_run_wall_seconds':time.perf_counter()-SETUP_STARTED,
            'independent_timestamp_attested':False,'research_only':True,'trade_ready':False}
    with (OUTPUT/('colab-cu-observation-'+SESSION+'-'+uuid.uuid4().hex[:8]+'.json')).open('x',encoding='utf-8') as f: json.dump(record,f,indent=2)
    final=checkpoint('final')
    print('交付完成：请把 final manifest 和结果包位置交给本地 Agent 独立审核。')
else:
    print('本地只读复核完成；没有访问 Colab，没有新增 fit，也没有云端 CU 证据。')